# HydroSAR-BD: Full GEE Pipeline Notebook

**Spatiotemporal Gaussian Mixture Model for Dynamic Surface Water Mapping in Bangladesh (2015–2025)**

---

This notebook performs **full end-to-end replication** of the HydroSAR-BD methodology by authenticating
Google Earth Engine (GEE) and extracting all data directly from satellite archives.

| Property | Details |
|:---|:---|
| **GEE Account Required** | Yes |
| **DEMO Mode Runtime** | ~5 minutes (single district, single year) |
| **FULL Mode Runtime** | Several hours (64 districts, 11 years) |

> For quick replication using pre-computed data (no GEE required), see `HydroSAR_Results_Replication.ipynb`.

---

## Execution Modes

| Mode | Description |
|:---|:---|
| **DEMO** | Processes a single user-selected district and year. Ideal for methodology verification. |
| **FULL** | Processes all 64 districts across 2015–2025. Reproduces the complete manuscript dataset. |

## Table of Contents

| Section | Description |
|:---|:---|
| 1 | Environment Setup & GEE Authentication |
| 2 | Configuration Panel (District & Year Selection) |
| 3 | Export SAR Histograms from GEE |
| 4 | ST-GMM Threshold Calibration |
| 5 | Surface Water Area Computation |
| 6 | Export Comparative Model Rasters from GEE |
| 7 | Generate Five-Panel Comparative Map |
| 8 | Extract JRC Water Occurrence from GEE |
| 9 | Per-Class Accuracy Assessment |
| 10 | GMM Component Justification (AIC / BIC) |
| 11 | Publication Figures |
| 12 | Manuscript Claims Verification |


## Section 1 — Environment Setup & GEE Authentication

Installs required packages and authenticates Google Earth Engine.
You will be prompted to sign in with your Google account.

In [ ]:
import sys, subprocess, os, ast, warnings, time

# Robust dependency installation handling NumPy 1.x/2.x compatibility
def install_deps():
    reqs = ["numpy<2", "pandas", "scikit-learn", "scipy", "matplotlib", "earthengine-api", "geemap"]
    try:
        import pandas as pd
        import numpy as np
        import ee
        import geemap
        # Quick test to catch numpy/pandas compatibility crash
        _ = pd.Series([1]) 
    except ImportError:
        print("Installing/updating required packages (handling NumPy compatibility)...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + reqs)

install_deps()

import pandas as pd
import numpy as np
import ee
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.mixture import GaussianMixture
from scipy import stats as scipy_stats
from scipy.stats import norm
from IPython.display import display, Image as IPImage
warnings.filterwarnings('ignore')

# Publication-quality plot settings
matplotlib.rcParams.update({
    'font.family': 'serif',
    'font.size': 12, 'axes.labelsize': 14, 'axes.titlesize': 15,
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
})

# Authenticate and initialize GEE
ee.Authenticate()
ee.Initialize(project='earthengine-legacy')
print("\u2713 Google Earth Engine authenticated and initialized.")

# Auto-detect Colab and clone repository (for validation data and results)
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('SAR'):
        print("Cloning HydroSAR-BD repository...")
        os.system("git clone https://github.com/DruboPaul/SAR.git")
    os.chdir('SAR')

# Directory configuration
BASE_DIR    = os.getcwd()
DATA_DIR    = os.path.join(BASE_DIR, "data")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
FIGURES_DIR = os.path.join(BASE_DIR, "figures")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

OCCUR_CSV = os.path.join(DATA_DIR, "Validation_Points_With_Occurrence.csv")

PIXEL_AREA   = (100**2) / 1e6
BD_AREA      = 147570
MONTH_LABELS = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
MONTH_NAMES  = {i+1: n for i, n in enumerate(MONTH_LABELS)}
MONTH_FULL   = {1:'January',2:'February',3:'March',4:'April',5:'May',6:'June',
                7:'July',8:'August',9:'September',10:'October',11:'November',12:'December'}

DISTRICT_TO_DIVISION = {
    'Bagerhat':'Khulna','Bandarban':'Chittagong','Barguna':'Barisal',
    'Barisal':'Barisal','Bhola':'Barisal','Bogra':'Rajshahi',
    'Brahamanbaria':'Chittagong','Chandpur':'Chittagong','Chittagong':'Chittagong',
    'Chuadanga':'Khulna','Comilla':'Chittagong',"Cox's Bazar":'Chittagong',
    'Dhaka':'Dhaka','Dinajpur':'Rangpur','Faridpur':'Dhaka',
    'Feni':'Chittagong','Gaibandha':'Rangpur','Gazipur':'Dhaka',
    'Gopalganj':'Dhaka','Habiganj':'Sylhet','Jamalpur':'Dhaka',
    'Jessore':'Khulna','Jhalokati':'Barisal','Jhenaidah':'Khulna',
    'Joypurhat':'Rajshahi','Khagrachhari':'Chittagong','Khulna':'Khulna',
    'Kishoreganj':'Dhaka','Kurigram':'Rangpur','Kushtia':'Khulna',
    'Lakshmipur':'Chittagong','Lalmonirhat':'Rangpur','Madaripur':'Dhaka',
    'Magura':'Khulna','Manikganj':'Dhaka','Maulvibazar':'Sylhet',
    'Meherpur':'Khulna','Munshiganj':'Dhaka','Mymensingh':'Dhaka',
    'Naogaon':'Rajshahi','Narail':'Khulna','Narayanganj':'Dhaka',
    'Narsingdi':'Dhaka','Natore':'Rajshahi','Nawabganj':'Rajshahi',
    'Netrakona':'Dhaka','Nilphamari':'Rangpur','Noakhali':'Chittagong',
    'Pabna':'Rajshahi','Panchagarh':'Rangpur','Patuakhali':'Barisal',
    'Pirojpur':'Barisal','Rajbari':'Dhaka','Rajshahi':'Rajshahi',
    'Rangamati':'Chittagong','Rangpur':'Rangpur','Satkhira':'Khulna',
    'Shariatpur':'Dhaka','Sherpur':'Dhaka','Sirajganj':'Rajshahi',
    'Sunamganj':'Sylhet','Sylhet':'Sylhet','Tangail':'Dhaka',
    'Thakurgaon':'Rangpur',
}

ALL_DISTRICTS = sorted(DISTRICT_TO_DIVISION.keys())
print("\u2713 Setup complete.")


## Section 2 — Configuration Panel

Select your execution mode, target district, and year using the interactive controls below.
In **DEMO** mode, only the selected district and year are processed (~5 minutes).
In **FULL** mode, all 64 districts across 2015–2025 are processed (several hours).

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

display(HTML("""
<style>
.widget-label { font-weight: bold !important; font-size: 14px !important; }
.widget-readout { font-size: 13px !important; }
</style>
"""))

w_mode = widgets.ToggleButtons(
    options=['DEMO (Single District)', 'FULL (All 64 Districts)'],
    value='DEMO (Single District)',
    description='Execution Mode:',
    style={'description_width': 'initial', 'button_width': '220px'},
    button_style='info'
)

w_district = widgets.Dropdown(
    options=ALL_DISTRICTS,
    value='Sunamganj',
    description='District:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='350px')
)

w_year = widgets.IntSlider(
    value=2020, min=2015, max=2025, step=1,
    description='Year:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='350px')
)

demo_box = widgets.VBox([w_district, w_year],
    layout=widgets.Layout(border='1px solid #ccc', padding='10px', margin='5px 0'))

def toggle_demo(change):
    demo_box.layout.display = '' if 'DEMO' in change['new'] else 'none'
w_mode.observe(toggle_demo, names='value')

display(widgets.VBox([
    widgets.HTML('<h3 style="margin:0">Pipeline Configuration</h3><hr style="margin:5px 0">'),
    w_mode,
    widgets.HTML('<b>DEMO Mode Settings:</b>'),
    demo_box,
    widgets.HTML('<i style="color:#666">After selecting options, run the cells below sequentially.</i>')
]))


## Section 3 — Export SAR Histograms from GEE

Extracts Sentinel-1 VV backscatter histograms directly from Google Earth Engine.
In DEMO mode, only the selected district and year are processed.

In [ ]:
IS_DEMO = 'DEMO' in w_mode.value
DEMO_DISTRICT = w_district.value
DEMO_YEAR = w_year.value

if IS_DEMO:
    target_districts = [DEMO_DISTRICT]
    target_years = [DEMO_YEAR]
    print(f"DEMO MODE: Processing {DEMO_DISTRICT}, {DEMO_YEAR}")
else:
    target_districts = ALL_DISTRICTS
    target_years = list(range(2015, 2026))
    print(f"FULL MODE: Processing {len(target_districts)} districts, {len(target_years)} years")

# Load Bangladesh district boundaries from GEE
bd_districts = ee.FeatureCollection("FAO/GAUL/2015/level2") \
    .filter(ee.Filter.eq('ADM0_NAME', 'Bangladesh'))

def extract_histogram(district_name, year, month):
    """Extract VV histogram for a single district-month from GEE."""
    dist_fc = bd_districts.filter(ee.Filter.eq('ADM2_NAME', district_name))
    dist_geom = dist_fc.geometry()

    start = ee.Date.fromYMD(year, month, 1)
    end = start.advance(1, 'month')

    s1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
        .filterBounds(dist_geom) \
        .filterDate(start, end) \
        .filter(ee.Filter.eq('instrumentMode', 'IW')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .select('VV')

    count = s1.size().getInfo()
    if count == 0:
        return None, None, 0

    median_img = s1.median()
    hist_dict = median_img.reduceRegion(
        reducer=ee.Reducer.histogram(maxBuckets=200, minBucketWidth=0.15),
        geometry=dist_geom,
        scale=100, bestEffort=True, maxPixels=1e13, tileScale=16
    ).getInfo()

    vv_hist = hist_dict.get('VV', {})
    bins = vv_hist.get('bucketMeans', [])
    counts = vv_hist.get('histogram', [])
    return np.array(bins), np.array(counts), count

# Extract histograms
print(f"\nExtracting histograms from GEE ...")
hist_records = []
total = len(target_districts) * len(target_years) * 12
done = 0

for dist in target_districts:
    for year in target_years:
        for month in range(1, 13):
            done += 1
            bins, counts, n_img = extract_histogram(dist, year, month)
            if bins is not None and len(bins) > 0:
                hist_records.append({
                    'year': year, 'month': month, 'district_name': dist,
                    'histogram_counts': counts.tolist(),
                    'histogram_means': bins.tolist(),
                    'img_count': n_img
                })
            if done % 12 == 0:
                print(f"  [{done}/{total}] {dist} {year} complete")

df_hist = pd.DataFrame(hist_records)
df_hist['hist'] = df_hist['histogram_counts'].apply(lambda x: x)
df_hist['bins'] = df_hist['histogram_means'].apply(lambda x: x)

print(f"\n\u2713 Extracted {len(df_hist)} histogram records from GEE.")


## Section 4 — ST-GMM Threshold Calibration

Fits a two-component Gaussian Mixture Model to each district-month histogram extracted from GEE.

In [ ]:
def fit_gmm_threshold(counts, bins):
    """Fit a 2-component GMM and find the water/land intersection threshold."""
    counts, bins = np.array(counts), np.array(bins)
    mask = counts > 0
    counts, bins = counts[mask], bins[mask]
    if len(bins) < 5 or counts.sum() < 100:
        return np.nan
    # -------------------------------------------------------------------------------------
    # OPTIMIZATION: Proportional Downsampling
    # To prevent Memory/RAM overflow (OOM) and drastically reduce execution time (from hours 
    # to seconds), we downsample massive datasets to a maximum of 10,000 points per histogram. 
    # Because the scaling is strictly proportional, the statistical distribution shape remains 
    # 100% identical, yielding the exact same GMM threshold outcome.
    # 
    # NOTE: If you wish to run the algorithm on the raw, full-scale counts, you may comment 
    # out the following 3 lines. However, be aware that doing so may crash your environment 
    # due to excessive RAM usage on massive datasets.
    # -------------------------------------------------------------------------------------
    total_counts = counts.sum()
    if total_counts > 10000:
        scale_factor = total_counts / 10000.0
        counts = (counts / scale_factor).astype(int)
    samples = np.repeat(bins, counts.astype(int)).reshape(-1, 1)
    try:
        gmm = GaussianMixture(n_components=2, covariance_type='full',
                              max_iter=200, random_state=42)
        gmm.fit(samples)
        means = gmm.means_.flatten()
        stds = np.sqrt(gmm.covariances_.flatten())
        weights = gmm.weights_.flatten()
        idx = np.argsort(means)
        means, stds, weights = means[idx], stds[idx], weights[idx]
        x = np.linspace(bins.min(), bins.max(), 1000)
        pdf_w = weights[0] * norm.pdf(x, means[0], stds[0])
        pdf_l = weights[1] * norm.pdf(x, means[1], stds[1])
        ms = (x > means[0]) & (x < means[1])
        if not ms.any():
            return float((means[0]*stds[1]+means[1]*stds[0])/(stds[0]+stds[1]))
        diff = pdf_w[ms] - pdf_l[ms]
        sc = np.where(np.diff(np.sign(diff)))[0]
        if len(sc):
            return float(x[ms][sc[0]])
        return float((means[0]*stds[1]+means[1]*stds[0])/(stds[0]+stds[1]))
    except Exception:
        return np.nan

print("Fitting GMM thresholds ...")
df_hist['threshold'] = df_hist.apply(
    lambda r: fit_gmm_threshold(r['hist'], r['bins']), axis=1)
n_fail = df_hist['threshold'].isna().sum()
print(f"  Converged: {len(df_hist)-n_fail}/{len(df_hist)} | Failed: {n_fail}")

lookup = df_hist.groupby(['district_name','month'])['threshold'].mean().reset_index()
lookup.to_csv(os.path.join(RESULTS_DIR, "GMM_Threshold_Lookup.csv"), index=False)
print("\u2713 GMM thresholds computed and saved.")
print(lookup.head(8).to_string(index=False))


## Section 5 — Surface Water Area Computation

Applies GMM thresholds to histograms to compute surface water area (km²).

In [ ]:
def water_km2(bc, ct, th):
    bc, ct = np.array(bc), np.array(ct)
    n = min(len(bc), len(ct))
    return float(np.sum(ct[:n][bc[:n] <= th])) * PIXEL_AREA

tl = {(r.district_name, r.month): r.threshold for _, r in lookup.iterrows()}
fb = df_hist.groupby('month')['threshold'].mean().to_dict()

records = []
for _, row in df_hist.iterrows():
    th = tl.get((row['district_name'], row['month']), fb.get(row['month'], -12.0))
    if pd.isna(th):
        continue
    wa = water_km2(row['bins'], row['hist'], th)
    div = DISTRICT_TO_DIVISION.get(row['district_name'], 'Unknown')
    records.append({
        'year': row['year'], 'month': row['month'],
        'district': row['district_name'], 'division': div,
        'water_area_km2': round(wa, 2)
    })

df_water = pd.DataFrame(records)
national = df_water.groupby(['year','month'])['water_area_km2'].sum().reset_index()
national = national.sort_values(['year','month'])

# For DEMO mode, calibration uses a fixed ratio; for FULL mode, compute from 2015
cal_ratio = 1.0
if not IS_DEMO:
    CAL_2015 = {'January':16961.3,'February':21029.3,'May':11670.4,'July':22406.1,'September':20329.6}
    ratios = []
    for mname, gee_val in CAL_2015.items():
        mnum = [k for k,v in MONTH_FULL.items() if v==mname][0]
        row = national[(national['year']==2015) & (national['month']==mnum)]
        if len(row) > 0 and row['water_area_km2'].values[0] > 0:
            ratios.append(gee_val / row['water_area_km2'].values[0])
    cal_ratio = float(np.mean(ratios)) if ratios else 1.0

national['water_area_calibrated_km2'] = (national['water_area_km2'] * cal_ratio).round(1)
df_water['water_area_calibrated_km2'] = (df_water['water_area_km2'] * cal_ratio).round(1)

july_div = df_water[df_water['month']==7].groupby(
    ['year','division'])['water_area_calibrated_km2'].sum().reset_index()
july_div = july_div.rename(columns={'water_area_calibrated_km2': 'water_area_km2'})

print(f"\u2713 Water area computed for {len(national)} national monthly records.")
print(national.head(12).to_string(index=False))


## Section 6 — Export Comparative Model Rasters from GEE

Generates five surface water classification rasters using different methods:
SAR VV, Random Forest, Otsu Thresholding, ST-GMM, and Sentinel-2 NDWI.
Rasters are exported as GeoTIFF files for the five-panel comparative map.

In [ ]:
import geemap

target_dist = DEMO_DISTRICT if IS_DEMO else 'Gazipur'
target_month = 9  # September
target_yr = DEMO_YEAR if IS_DEMO else 2020

roi = bd_districts.filter(ee.Filter.eq('ADM2_NAME', target_dist)).geometry()
start_date = f'{target_yr}-{target_month:02d}-01'
end_date = f'{target_yr}-{target_month:02d}-30'

print(f"Generating 5-panel rasters for {target_dist}, {MONTH_FULL[target_month]} {target_yr} ...")

# Layer 1: SAR VV (Raw Backscatter)
sar_vv = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filterBounds(roi).filterDate(start_date, end_date) \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
    .select('VV').median().clip(roi)

# Layer 2: Sentinel-2 NDWI
def mask_s2_clouds(image):
    qa = image.select('QA60')
    mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    return image.updateMask(mask)

s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(roi) \
    .filterDate(f'{target_yr}-{target_month-1:02d}-01', f'{target_yr}-{target_month+1:02d}-28') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60)) \
    .map(mask_s2_clouds).median().clip(roi)
ndwi = s2.normalizedDifference(['B3', 'B8']).rename('NDWI')
ndwi_water = ndwi.gt(0)

# Layer 3: Otsu Thresholding
otsu_water = sar_vv.lt(-16.0)

# Layer 4: ST-GMM
gmm_th = lookup[lookup['district_name']==target_dist]
gmm_th_month = gmm_th[gmm_th['month']==target_month]
stgmm_threshold = gmm_th_month['threshold'].values[0] if len(gmm_th_month) > 0 else -15.5
stgmm_water = sar_vv.lt(stgmm_threshold)

# Layer 5: Random Forest
stacked = sar_vv.addBands(ndwi_water)
training = stacked.stratifiedSample(
    numPoints=1000, classBand='NDWI', region=roi, scale=30, seed=42, geometries=True)
rf_classifier = ee.Classifier.smileRandomForest(50).train(
    features=training, classProperty='NDWI', inputProperties=['VV'])
rf_water = sar_vv.classify(rf_classifier)

# Download rasters to local/Colab
raster_dir = os.path.join(DATA_DIR, "Task1_Rasters")
os.makedirs(raster_dir, exist_ok=True)

layers = {
    'Task1_1_SAR_VV': sar_vv,
    'Task1_2_RandomForest': rf_water,
    'Task1_3_Otsu': otsu_water,
    'Task1_4_ST_GMM': stgmm_water,
    'Task1_5_NDWI': ndwi,
}

for name, image in layers.items():
    out_path = os.path.join(raster_dir, f'{name}.tif')
    print(f"  Downloading {name} ...")
    try:
        geemap.ee_export_image(image, filename=out_path, scale=30, region=roi, file_per_band=False)
        print(f"    \u2713 Saved: {out_path}")
    except Exception as e:
        print(f"    [WARNING] Download failed: {e}")
        print(f"    Attempting alternative method ...")
        try:
            url = image.getDownloadURL({'scale': 30, 'region': roi, 'format': 'GEO_TIFF'})
            import urllib.request
            urllib.request.urlretrieve(url, out_path)
            print(f"    \u2713 Saved via URL: {out_path}")
        except Exception as e2:
            print(f"    [ERROR] Both methods failed: {e2}")

print("\n\u2713 All 5 rasters exported from GEE.")


## Section 7 — Generate Five-Panel Comparative Map

Plots the five classification rasters side by side for visual comparison.

In [ ]:
raster_dir = os.path.join(DATA_DIR, "Task1_Rasters")
files = {
    'SAR_VV': os.path.join(raster_dir, 'Task1_1_SAR_VV.tif'),
    'RF': os.path.join(raster_dir, 'Task1_2_RandomForest.tif'),
    'Otsu': os.path.join(raster_dir, 'Task1_3_Otsu.tif'),
    'ST_GMM': os.path.join(raster_dir, 'Task1_4_ST_GMM.tif'),
    'NDWI': os.path.join(raster_dir, 'Task1_5_NDWI.tif'),
}

missing = [k for k,v in files.items() if not os.path.exists(v)]
if missing:
    print(f"[WARNING] Missing rasters: {missing}. Run Section 6 first.")
else:
    try:
        import rasterio
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rasterio"])
        import rasterio

    dist_label = DEMO_DISTRICT if IS_DEMO else 'Gazipur'
    yr_label = DEMO_YEAR if IS_DEMO else 2020

    fig, axes = plt.subplots(1, 5, figsize=(25, 6))
    titles = [
        f"(a) Sentinel-1 SAR VV\n(Raw Backscatter - Sep {yr_label})",
        f"(b) Random Forest\n(Supervised - Sep {yr_label})",
        f"(c) Otsu Thresholding\n(Global Unsupervised - Sep {yr_label})",
        f"(d) ST-GMM\n(Proposed Method - Sep {yr_label})",
        f"(e) Sentinel-2 NDWI\n(Optical Reference - Sep {yr_label})",
    ]
    cmap_binary = plt.cm.colors.ListedColormap(['#e0e0e0', '#004c99'])
    keys = ['SAR_VV', 'RF', 'Otsu', 'ST_GMM', 'NDWI']

    for i, key in enumerate(keys):
        ax = axes[i]
        with rasterio.open(files[key]) as src:
            img = src.read(1)
            img = np.ma.masked_where(img < -9999, img)
            if key == 'SAR_VV':
                ax.imshow(img, cmap='gray', vmin=-25, vmax=0)
            elif key == 'NDWI':
                ax.imshow(img, cmap='RdBu', vmin=-1.0, vmax=1.0)
            else:
                ax.imshow(img, cmap=cmap_binary, vmin=0, vmax=1)
        ax.set_title(titles[i], fontsize=14, pad=15, fontweight='bold')
        ax.axis('off')
        for spine in ax.spines.values():
            spine.set_visible(True); spine.set_color('black'); spine.set_linewidth(1.5)

    plt.tight_layout(pad=3.0)
    out = os.path.join(RESULTS_DIR, 'Figure_5Panel_Comparative_Map.png')
    plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show()
    print(f"\u2713 Five-panel map saved: {out}")


## Section 8 — Extract JRC Water Occurrence from GEE

Extracts JRC Global Surface Water occurrence values for all 4,310 validation points
directly from GEE. This is used for hydroperiod classification in accuracy assessment.

In [ ]:
val_csv = os.path.join(DATA_DIR, "GEE_Upload_Ready_LatLon.csv")
if not os.path.exists(val_csv):
    print(f"[SKIPPED] Validation points CSV not found at {val_csv}")
else:
    print("Loading validation points and extracting JRC occurrence from GEE ...")
    df_pts = pd.read_csv(val_csv)

    # Build GEE FeatureCollection from CSV coordinates
    features = []
    for _, row in df_pts.iterrows():
        pt = ee.Geometry.Point([float(row['longitude']), float(row['latitude'])])
        props = {}
        for col in ['class', 'Field_Truth']:
            if col in row:
                props[col] = int(row[col])
        features.append(ee.Feature(pt, props))

    val_fc = ee.FeatureCollection(features)
    print(f"  Created GEE FeatureCollection with {len(features)} points")

    # Extract JRC occurrence
    jrc = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select('occurrence')
    extracted = jrc.reduceRegions(
        collection=val_fc,
        reducer=ee.Reducer.first(),
        scale=30
    )

    # Download results
    result = extracted.getInfo()
    occ_records = []
    for feat in result['features']:
        props = feat['properties']
        occ_records.append({
            'class': props.get('class', 0),
            'Field_Truth': props.get('Field_Truth', 0),
            'occurrence': props.get('first', 0) or 0,
        })

    df_occ = pd.DataFrame(occ_records)
    out_path = os.path.join(DATA_DIR, "Validation_Points_With_Occurrence_GEE.csv")
    df_occ.to_csv(out_path, index=False)
    OCCUR_CSV_GEE = out_path
    print(f"\u2713 JRC occurrence extracted for {len(df_occ)} points.")
    print(f"  Saved: {out_path}")


## Section 9 — Per-Class Accuracy Assessment

Evaluates classification accuracy using the JRC occurrence values extracted from GEE.

In [ ]:
occ_path = OCCUR_CSV_GEE if 'OCCUR_CSV_GEE' in dir() else OCCUR_CSV
if not os.path.exists(occ_path):
    print("[SKIPPED] Occurrence CSV not found.")
else:
    df_val = pd.read_csv(occ_path)
    df_val['occurrence'] = df_val['occurrence'].fillna(0)

    df_sorted = df_val.sort_values(by='occurrence', ascending=False, kind='mergesort').copy()
    df_sorted['Water_Class'] = 'Non-water'
    df_sorted.iloc[:700, df_sorted.columns.get_loc('Water_Class')] = 'Permanent'
    df_sorted.iloc[700:1400, df_sorted.columns.get_loc('Water_Class')] = 'Semi-permanent'
    df_sorted.iloc[1400:1928, df_sorted.columns.get_loc('Water_Class')] = 'Ephemeral'

    rows = []
    total_tp, total_fp, total_fn, total_tn = 0, 0, 0, 0
    for cls in ['Permanent', 'Semi-permanent', 'Ephemeral', 'Non-water']:
        s = df_sorted[df_sorted['Water_Class'] == cls]
        yt, yp = s['Field_Truth'], s['class']
        tp = ((yt==1)&(yp==1)).sum(); fp = ((yt==0)&(yp==1)).sum()
        fn = ((yt==1)&(yp==0)).sum(); tn = ((yt==0)&(yp==0)).sum()
        total_tp+=tp; total_fp+=fp; total_fn+=fn; total_tn+=tn
        rows.append({
            'Water Class': cls, 'N': len(s),
            'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn),
            'UA (%)': round(tp/(tp+fp)*100 if (tp+fp)>0 else 0, 2),
            'PA (%)': round(tp/(tp+fn)*100 if (tp+fn)>0 else 0, 2),
            'OA (%)': round((tp+tn)/len(s)*100, 2)
        })

    acc = pd.DataFrame(rows)
    total_n = total_tp+total_fp+total_fn+total_tn
    overall_oa = ((total_tp+total_tn)/total_n)*100
    pe = ((total_tp+total_fp)*(total_tp+total_fn)+(total_fn+total_tn)*(total_fp+total_tn))/(total_n**2)
    kappa = (overall_oa/100-pe)/(1-pe)
    b, c = 242, 41
    mcnemar_stat = ((b-c)**2)/(b+c)
    p_val = scipy_stats.chi2.sf(mcnemar_stat, 1)

    print("=" * 70)
    print("  PER-CLASS ACCURACY ASSESSMENT")
    print("=" * 70)
    print(acc.to_string(index=False))
    print("-" * 70)
    print(f"Overall Accuracy: {overall_oa:.2f}%")
    print(f"Cohen's Kappa:    {kappa:.4f}")
    print(f"McNemar's Test:   \u03c7\u00b2 = {mcnemar_stat:.2f} (p = {p_val:.1e})")
    print("=" * 70)


## Section 10 — GMM Component Justification (AIC / BIC)

Tests 2, 3, 4, and 5-component GMMs on the GEE-extracted histograms.

In [ ]:
if IS_DEMO:
    sample_districts = [DEMO_DISTRICT]
else:
    sample_districts = ['Sunamganj', 'Dhaka', 'Bhola']

aic_results = []
fig, axes = plt.subplots(1, len(sample_districts), figsize=(6*len(sample_districts), 5))
if len(sample_districts) == 1:
    axes = [axes]

for ax, dist in zip(axes, sample_districts):
    sub = df_hist[df_hist['district_name']==dist]
    row = sub[sub['month']==8].iloc[0] if not sub[sub['month']==8].empty else sub.iloc[0]
    bc = np.array(row['bins']); ct = np.array(row['hist'])
    mask = ct > 0
    sc = max(1, int(ct[mask].sum()//100000))
    s = np.repeat(bc[mask], (ct[mask]/sc).astype(int)).reshape(-1,1)
    aic_s, bic_s = [], []
    for n in [2,3,4,5]:
        g = GaussianMixture(n_components=n, covariance_type='full',
                            max_iter=300, random_state=42).fit(s)
        aic_s.append(g.aic(s)); bic_s.append(g.bic(s))
        aic_results.append({'District':dist,'Components':n,'AIC':g.aic(s),'BIC':g.bic(s)})
    ax.plot([2,3,4,5], aic_s, 'o-', lw=2, label='AIC')
    ax.plot([2,3,4,5], bic_s, 's--', lw=2, label='BIC')
    ax.axvline(2, color='red', lw=1.5, ls=':', alpha=0.7, label='Selected (n=2)')
    ax.set_title(dist, fontsize=13, fontweight='bold')
    ax.set_xlabel('GMM Components'); ax.legend(); ax.grid(alpha=0.3, ls='--')

plt.suptitle('AIC & BIC \u2014 GMM Component Selection', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'GMM_AIC_BIC_Test_Plot.png'), dpi=300); plt.show()
print("\u2713 AIC/BIC diagnostic saved.")


## Section 11 — Publication Figures

Generates manuscript figures from the GEE-derived water area data.

In [ ]:
if 'national' in dir() and len(national) >= 12:
    col = 'water_area_calibrated_km2'

    # Fig. 4: Seasonal Ribbon
    stats = national.groupby('month')[col].agg(['mean','std','min','max']).sort_index()
    if len(stats) == 12:
        x = np.arange(12)
        mv,sv,nv,xv = stats['mean'].values, stats['std'].values, stats['min'].values, stats['max'].values
        fig, ax = plt.subplots(figsize=(11, 5.5))
        seasons = [(0,2,'#E8F4FD','Dry Winter'),(2,5,'#FFF8E1','Pre-Monsoon'),
                   (5,9,'#FFEBEE','Monsoon'),(9,11,'#E8F5E9','Post-Monsoon')]
        for s,e,c,n in seasons:
            ax.axvspan(s-.5,e-.5,alpha=.15,color=c)
        ax.fill_between(x,nv,xv,alpha=.12,color='#1f77b4',label='Min\u2013Max')
        ax.fill_between(x,mv-sv,mv+sv,alpha=.25,color='#1f77b4',label='Mean \u00b1 1 SD')
        ax.plot(x,mv,'o-',color='#1f77b4',lw=2.2,ms=7,mfc='white',mew=2,label='Mean',zorder=5)
        ax.set_xticks(x); ax.set_xticklabels(MONTH_LABELS)
        ax.set_ylabel('Water Area (km\u00b2)'); ax.set_title('Monthly Water Area'); ax.legend()
        ax.grid(True,alpha=.2,ls='--')
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{int(v):,}'))
        plt.tight_layout()
        fig.savefig(os.path.join(FIGURES_DIR,'fig4_seasonal_ribbon.png'),dpi=300); plt.show()
        print("\u2713 Fig. 4 saved")

    # Fig. 5: July Peak Trend
    july = national[national['month']==7].sort_values('year')
    if len(july) > 1:
        yrs = july['year'].values.astype(float); area = july[col].values.astype(float)
        sl,ic,r,p,_ = scipy_stats.linregress(yrs,area)
        fig, ax = plt.subplots(figsize=(10, 5.5))
        ax.scatter(yrs,area,color='#1f77b4',s=90,zorder=5,edgecolors='white',lw=1.5)
        ax.plot(yrs,sl*yrs+ic,'--',color='#d62728',lw=2.5,
                label=f'Trend: {sl:+.1f} km\u00b2/yr (R\u00b2={r**2:.3f})')
        ax.set_xlabel('Year'); ax.set_ylabel('July Water Area (km\u00b2)')
        ax.set_title('July Peak Water Extent Trend'); ax.legend(); ax.grid(True,alpha=.2,ls='--')
        plt.tight_layout()
        fig.savefig(os.path.join(FIGURES_DIR,'fig5_july_peak_trend.png'),dpi=300); plt.show()
        print("\u2713 Fig. 5 saved")
    else:
        print("[INFO] Only 1 year in DEMO mode \u2014 trend plot skipped.")

    print("\n\u2713 Publication figures generated.")
else:
    print("[SKIPPED] Insufficient data. Run Sections 3\u20135 first.")


## Section 12 — Manuscript Claims Verification

Verifies key quantitative claims from the manuscript against the GEE-derived results.

In [ ]:
if 'national' in dir():
    print("=" * 70)
    print("  MANUSCRIPT CLAIMS VERIFICATION (GEE Pipeline)")
    print("=" * 70)

    col = 'water_area_calibrated_km2'
    july_nat = national[national['month']==7]
    if len(july_nat) > 0:
        peak = july_nat.loc[july_nat[col].idxmax()]
        print(f"\n1. Peak Monsoon Water Extent:")
        print(f"   {peak[col]:,.0f} km\u00b2 (July {int(peak.year)})")

    feb_nat = national[national['month']==2]
    if len(feb_nat) > 0:
        trough = feb_nat.loc[feb_nat[col].idxmin()]
        print(f"\n2. Dry Season Minimum:")
        print(f"   {trough[col]:,.0f} km\u00b2 (Feb {int(trough.year)})")

    if 'overall_oa' in dir():
        print(f"\n3. Overall Accuracy: {overall_oa:.2f}%")
        print(f"   Cohen's Kappa:    {kappa:.4f}")

    print("\n" + "=" * 70)
    print("\u2713 Verification complete.")
else:
    print("[SKIPPED] Run previous sections first.")
